In [15]:
import pickle
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from evaluator import *


with open('data-challenge-student.pickle', 'rb') as handle:
    # dat = pickle.load(handle)
    dat = pd.read_pickle(handle)
X = dat['X_train']
Y = dat['Y']
S = dat['S_train']

In [16]:
X_train, X_test, Y_train, Y_test, S_train, S_test = train_test_split(X, Y, S, test_size=0.2, random_state=42)

In [17]:
from sklearn.utils import resample
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# 将S、X和Y合并为一个DataFrame
data = pd.concat([ X,S, Y], axis=1)

#data_test = pd.concat([S_test, X_test], axis=1)

In [18]:
# 分割数据为S=0和S=1的子集
data_s0 = data[data['gender_class'] == 0]
data_s1 = data[data['gender_class'] == 1]

# 对S=1的子集进行重采样，使其数量与S=0的子集相等
data_s1_resampled = resample(data_s1, replace=True, n_samples=len(data_s0), random_state=42)

# 合并重采样后的S=1子集与S=0子集
balanced_data = pd.concat([data_s0, data_s1_resampled])

# 分离特征向量X和目标Y
X_balanced = balanced_data.drop(['profession_class'], axis=1)
Y_balanced = balanced_data['profession_class']


In [19]:
balanced_data_list = []
for nbclass in range(28):
    data_class = data[data['profession_class'] == nbclass]
    data_s0 = data_class[data_class['gender_class'] == 0]
    data_s1 = data_class[data_class['gender_class'] == 1]
    if len(data_s0)>=len(data_s1):
    # 对S=1的子集进行重采样，使其数量与S=0的子集相等
        data_s1_resampled = resample(data_s1, replace=True, n_samples=len(data_s0), random_state=42)

         # 合并重采样后的S=1子集与S=0子集
        balanced_data = pd.concat([data_s0, data_s1_resampled])
    else:
        data_s0_resampled = resample(data_s0, replace=True, n_samples=len(data_s1), random_state=42)

         # 合并重采样后的S=1子集与S=0子集
        balanced_data = pd.concat([data_s1, data_s0_resampled])
        # 将平衡后的数据添加到列表中
    balanced_data_list.append(balanced_data)

# 将所有平衡后的数据合并成一个DataFrame
balanced_data_combined = pd.concat(balanced_data_list)
        

    # 分离特征向量X和目标Y
X_balanced = balanced_data_combined.drop(['profession_class'], axis=1)
X_balanced = X_balanced.iloc[:, :-1]
Y_balanced = balanced_data_combined['profession_class']
S_balanced = balanced_data_combined['gender_class']

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold
from modelization import *
# Configuration de la validation croisée
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)

# Initialisation du modèle avec les paramètres spécifiques
model = LogisticRegression(random_state=42, solver='lbfgs', C=0.1, multi_class='multinomial', max_iter=5000)
result = train_and_evaluate(model, X_balanced, Y_balanced, S_balanced, cv)






In [ ]:
X_true = dat['X_test']
S_true = dat['S_test'] 
data_true = pd.concat([S_true, X_true], axis=1)
# Classify the provided test data with you classifier
regression_classique = result[0].predict(X_test.values)
results=pd.DataFrame(regression_classique, columns= ['score'])
results.to_csv("Data_Challenge_MDI_9.csv", header = None, index = None)